# Flood–Weather Correlation Analysis

This notebook analyzes the association between historical flood events in Bhutan and ERA5 weather variables.
We will:
- Harmonize timezones (Asia/Thimphu)
- Aggregate ERA5 to daily values
- Map floods to nearest ERA5 grid cell
- Build event vs. control dataset
- Engineer antecedent features
- Explore correlations and simple models

## 1. Imports & Setup

In [1]:

import numpy as np
import pandas as pd

from scipy.stats import pointbiserialr, spearmanr
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

pd.set_option("display.max_columns", 100)

# Work on copies (assuming flood_df and era_df are already loaded)
flood = flood_df.copy()
era = era_df.copy()

flood.shape, era.shape


NameError: name 'flood_df' is not defined

## 2. Timezone Alignment
Ensure both datasets use Asia/Thimphu timezone.

In [ ]:

# ERA5 datetimes (already tz-aware, but confirm)
era['datetime'] = pd.to_datetime(era['datetime']).dt.tz_convert('Asia/Thimphu')

# Flood dates are tz-naive; localize to Asia/Thimphu
flood['Date'] = pd.to_datetime(flood['Date']).dt.tz_localize('Asia/Thimphu')

# Convenience daily dates
era['date'] = era['datetime'].dt.floor('D')
flood['date'] = flood['Date'].dt.floor('D')

era[['datetime','date']].head(), flood[['Date','date']].head()


## 3. Daily ERA5 Aggregation

In [ ]:

# Wind speed
era['wind_speed'] = np.sqrt(era['wind_u']**2 + era['wind_v']**2)

# Aggregate per grid cell per day
agg = {
    'precipitation': ['sum', 'max'],
    'surface_runoff': ['sum', 'mean'],
    'temperature': ['mean', 'max'],
    'wind_speed': ['mean', 'max']
}

daily_era = era.groupby(['latitude','longitude','date']).agg(agg)
daily_era.columns = ['_'.join(col) for col in daily_era.columns]
daily_era = daily_era.reset_index()

daily_era.rename(columns={
    'precipitation_sum':'P_0d',
    'precipitation_max':'P_max1h_0d',
    'surface_runoff_sum':'R_0d',
    'surface_runoff_mean':'R_mean_0d',
    'temperature_mean':'T_mean_0d',
    'temperature_max':'T_max_0d',
    'wind_speed_mean':'WS_mean_0d',
    'wind_speed_max':'WS_max_0d'
}, inplace=True)

daily_era.head()


## 4. Map Floods to Nearest ERA5 Grid Cell

In [ ]:

grid = daily_era[['latitude','longitude']].drop_duplicates().reset_index(drop=True)
grid_np = grid[['latitude','longitude']].to_numpy()

def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0
    lat1 = np.radians(lat1); lon1 = np.radians(lon1)
    lat2 = np.radians(lat2); lon2 = np.radians(lon2)
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2.0)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2.0)**2
    return 2*R*np.arcsin(np.sqrt(a))

def nearest_grid(lat, lon):
    diffs = grid_np - np.array([lat, lon])
    idx = np.argmin((diffs**2).sum(axis=1))
    d_km = haversine_km(lat, lon, grid_np[idx,0], grid_np[idx,1])
    return idx, d_km

mapped = flood[['EventID','Date','date','District','Location','Latitude','Longitude']].copy()
idxs = []; dists = []
for lat, lon in mapped[['Latitude','Longitude']].to_numpy():
    i, d = nearest_grid(lat, lon)
    idxs.append(i); dists.append(d)

mapped['grid_index'] = idxs
mapped['distance_km'] = dists
mapped = mapped.merge(grid.assign(grid_index=grid.index), on='grid_index', how='left') \
               .rename(columns={'latitude':'grid_lat','longitude':'grid_lon'})
mapped.head()


## 5. Deduplicate Events

In [ ]:

events = mapped.drop_duplicates(subset=['grid_lat','grid_lon','date']).copy()
events['flood'] = 1
events.shape, events.head()


## 6. Build Control Sample

In [ ]:

daily_era['year'] = daily_era['date'].dt.year
daily_era['month'] = daily_era['date'].dt.month

event_keys = set(zip(events['grid_lat'], events['grid_lon'], events['date']))
event_df = events[['grid_lat','grid_lon','date']].copy()
event_df['year'] = event_df['date'].dt.year
event_df['month'] = event_df['date'].dt.month

candidates = daily_era.groupby(['grid_lat','grid_lon','month'], as_index=False) \
    .apply(lambda g: g['date'].tolist()).rename(columns={0:'dates'})

K = 3
ctrl_rows = []
for lat, lon, dt, yr, mo in event_df[['grid_lat','grid_lon','date','year','month']].itertuples(index=False, name=None):
    cand_list = candidates.loc[
        (candidates['grid_lat']==lat) &
        (candidates['grid_lon']==lon) &
        (candidates['month']==mo), 'dates']
    if len(cand_list)==0:
        continue
    pool = [d for d in cand_list.iloc[0] if (d.year != yr) and ((lat,lon,d) not in event_keys)]
    if len(pool)==0:
        continue
    take = np.random.choice(pool, size=min(K, len(pool)), replace=False)
    for d in take:
        ctrl_rows.append((lat, lon, d, 0))

controls = pd.DataFrame(ctrl_rows, columns=['grid_lat','grid_lon','date','flood'])
controls.shape, controls.head()


## 7. Create Analysis Panel

In [ ]:

panel = pd.concat([events[['grid_lat','grid_lon','date','flood']],
                  controls[['grid_lat','grid_lon','date','flood']]], ignore_index=True)

panel.drop_duplicates(subset=['grid_lat','grid_lon','date','flood'], inplace=True)
panel['month'] = panel['date'].dt.month
panel['year'] = panel['date'].dt.year

panel.shape, panel['flood'].value_counts()


## 8. Feature Engineering

In [ ]:

fe = daily_era[['grid_lat','grid_lon','date','P_0d','P_max1h_0d','R_0d','R_mean_0d','T_mean_0d','WS_mean_0d','WS_max_0d']].copy()
fe = fe.sort_values(['grid_lat','grid_lon','date'])
g = fe.groupby(['grid_lat','grid_lon'], group_keys=False)

fe['P_1d'] = g['P_0d'].shift(1)
fe['P_3d'] = g['P_0d'].shift(1).rolling(3, min_periods=1).sum()
fe['P_7d'] = g['P_0d'].shift(1).rolling(7, min_periods=1).sum()

fe['R_1d'] = g['R_0d'].shift(1)
fe['R_3d'] = g['R_0d'].shift(1).rolling(3, min_periods=1).sum()
fe['R_7d'] = g['R_0d'].shift(1).rolling(7, min_periods=1).sum()

fe['T_mean_7d'] = g['T_mean_0d'].shift(1).rolling(7, min_periods=1).mean()

panel_feats = panel.merge(fe, on=['grid_lat','grid_lon','date'], how='left')
panel_feats.head()


## 9. Correlation Analysis

In [ ]:

feature_cols = ['P_0d','P_max1h_0d','P_1d','P_3d','P_7d','R_0d','R_mean_0d','R_1d','R_3d','R_7d','T_mean_0d','T_mean_7d','WS_mean_0d','WS_max_0d']

corr_rows = []
for col in feature_cols:
    s = panel_feats[col]
    mask = s.notna() & panel_feats['flood'].notna()
    if mask.sum() < 20:
        corr_rows.append((col, np.nan, np.nan, mask.sum()))
        continue
    r_pb, _ = pointbiserialr(panel_feats.loc[mask, 'flood'], s[mask])
    r_sp, _ = spearmanr(panel_feats.loc[mask, 'flood'], s[mask])
    corr_rows.append((col, r_pb, r_sp, mask.sum()))

corr_table = pd.DataFrame(corr_rows, columns=['feature','point_biserial_r','spearman_r','n'])
corr_table.sort_values('point_biserial_r', key=lambda s: s.abs(), ascending=False)


## 10. Logistic Regression Model

In [ ]:

model_features = ['P_0d','P_3d','P_7d','R_3d','T_mean_7d','WS_max_0d']

dfm = panel_feats[['flood','month'] + model_features].dropna().copy()
X = pd.get_dummies(dfm[['month'] + model_features], columns=['month'], drop_first=True)
y = dfm['flood'].astype(int)

cont_cols = model_features
scaler = StandardScaler()
X_scaled = X.copy()
X_scaled[cont_cols] = scaler.fit_transform(X[cont_cols])

logit = LogisticRegression(penalty='l2', C=1.0, solver='liblinear', class_weight='balanced', max_iter=200)
logit.fit(X_scaled, y)

coefs = pd.Series(logit.coef_[0], index=X_scaled.columns)
odds = np.exp(coefs)

odds_table = pd.DataFrame({'feature': X_scaled.columns, 'coef': coefs, 'odds_ratio': odds}) \
    .sort_values('coef', key=lambda s: s.abs(), ascending=False)

odds_table.head(20)
